# Playground Series S6E8 — Phase 1 reconnaissance
Local-data-only, modeling-relevant EDA. The official metric is not present in the supplied CSVs; ROC AUC is the working assumption because the submission requires probabilities.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from kaggle_smartphone_addiction import (
    CVConfig, build_baseline_model, cross_validate_model,
    evaluate_predictions, load_competition_data, split_features_target,
)
train, test, sample = load_competition_data(ROOT / 'data/raw')
X, y, X_test = split_features_target(train, test)

## Schema and problem type

In [2]:
pd.DataFrame({
    'file': ['train.csv', 'test.csv', 'sample_submission.csv'],
    'rows': [len(train), len(test), len(sample)],
    'columns': [train.shape[1], test.shape[1], sample.shape[1]],
}), train.dtypes.rename('dtype').to_frame(), sample.head()

(                    file    rows  columns
 0              train.csv  691369       14
 1               test.csv  296302       13
 2  sample_submission.csv  296302        2,
                            dtype
 id                         int64
 age                      float64
 daily_screen_time_hours  float64
 social_media_hours       float64
 gaming_hours             float64
 work_study_hours         float64
 sleep_hours              float64
 notifications_per_day    float64
 app_opens_per_day        float64
 weekend_screen_time      float64
 gender                       str
 stress_level                 str
 academic_work_impact         str
 addicted_label             int64,
        id  addicted_label
 0  691369        0.709424
 1  691370        0.709424
 2  691371        0.709424
 3  691372        0.709424
 4  691373        0.709424)

In [3]:
target_summary = train['addicted_label'].value_counts().sort_index().rename('count').to_frame()
target_summary['proportion'] = target_summary['count'] / len(train)
target_summary

,count,proportion
addicted_label,,
0,200895,0.290576
1,490474,0.709424


Binary 0/1 target + probability-shaped sample submission implies binary probabilistic classification. `id` is excluded from features.

## Missingness and cardinality

In [4]:
profile = pd.DataFrame({
    'dtype': X.dtypes.astype(str),
    'n_unique_train': X.nunique(dropna=True),
    'missing_train': X.isna().mean(),
    'missing_test': X_test.isna().mean(),
})
profile['missing_shift_pp'] = 100 * (profile['missing_test'] - profile['missing_train'])
profile

,dtype,n_unique_train,missing_train,missing_test,missing_shift_pp
age,float64,18,0.041843,0.057840,1.599657
daily_screen_time_hours,float64,1389,0.138644,0.110657,-2.798639
social_media_hours,float64,721,0.193811,0.159962,-3.384932
gaming_hours,float64,401,0.183435,0.200539,1.710403
work_study_hours,float64,600,0.074516,0.093746,1.922965
sleep_hours,float64,451,0.064336,0.075784,1.144804
notifications_per_day,float64,231,0.097754,0.115494,1.773978
app_opens_per_day,float64,166,0.116739,0.086753,-2.998669
weekend_screen_time,float64,1437,0.162089,0.171099,0.901053
gender,str,3,0.041995,0.047965,0.596964


All 12 model features contain missing values. The three categoricals are low-cardinality; there are no high-cardinality categoricals. Missingness shifts are more notable than value-distribution shifts and missing indicators are retained in the baseline.

## Duplicates and train/test drift

In [5]:
features = list(X.columns)
duplicate_checks = pd.Series({
    'duplicate_train_rows': int(train.duplicated().sum()),
    'duplicate_test_rows': int(test.duplicated().sum()),
    'duplicate_train_ids': int(train['id'].duplicated().sum()),
    'duplicate_test_ids': int(test['id'].duplicated().sum()),
    'overlapping_ids': len(set(train['id']) & set(test['id'])),
    'duplicate_train_feature_rows': int(train[features].duplicated().sum()),
    'duplicate_test_feature_rows': int(test[features].duplicated().sum()),
    'exact_cross_split_feature_matches': len(train[features].merge(test[features], how='inner')),
})
duplicate_checks

duplicate_train_rows                 0
duplicate_test_rows                  0
duplicate_train_ids                  0
duplicate_test_ids                   0
overlapping_ids                      0
duplicate_train_feature_rows         0
duplicate_test_feature_rows          0
exact_cross_split_feature_matches    2
dtype: int64

In [6]:
numeric = X.select_dtypes(include='number').columns
numeric_drift = pd.DataFrame(index=numeric)
numeric_drift['train_mean'] = X[numeric].mean()
numeric_drift['test_mean'] = X_test[numeric].mean()
numeric_drift['ks_statistic'] = [ks_2samp(X[c].dropna(), X_test[c].dropna()).statistic for c in numeric]
numeric_drift.sort_values('ks_statistic', ascending=False)

,train_mean,test_mean,ks_statistic
notifications_per_day,145.894900,145.748066,0.002683
sleep_hours,6.804334,6.801704,0.002180
gaming_hours,1.459265,1.457668,0.002089
app_opens_per_day,102.636781,102.656216,0.002083
daily_screen_time_hours,7.640865,7.640771,0.001727
weekend_screen_time,9.479866,9.474575,0.001671
social_media_hours,2.471038,2.471224,0.001599
age,26.615408,26.608689,0.001448
work_study_hours,2.366971,2.365962,0.000915


In [7]:
categorical = X.select_dtypes(exclude='number').columns
cat_drift = {}
for col in categorical:
    a = X[col].fillna('<MISSING>').value_counts(normalize=True)
    b = X_test[col].fillna('<MISSING>').value_counts(normalize=True)
    cat_drift[col] = 0.5 * sum(abs(a.get(v, 0) - b.get(v, 0)) for v in set(a.index) | set(b.index))
pd.Series(cat_drift, name='total_variation_distance').sort_values(ascending=False)

academic_work_impact    0.022841
stress_level            0.013530
gender                  0.005970
Name: total_variation_distance, dtype: float64

Value distributions are extremely close (all numeric KS statistics below 0.003; all categorical total-variation distances below 0.023). Two exact cross-split feature matches exist, but this is negligible at this scale.

## Signal and leakage review

In [8]:
train.select_dtypes(include='number').corr()['addicted_label'].sort_values(key=abs, ascending=False).to_frame('target_correlation')

,target_correlation
addicted_label,1.000000
daily_screen_time_hours,0.611398
weekend_screen_time,0.589903
social_media_hours,0.532409
work_study_hours,0.251416
gaming_hours,0.205283
app_opens_per_day,0.063482
sleep_hours,0.042545
notifications_per_day,-0.011583
age,0.004043


Potential risks: (1) `id` may encode generation order, so it is excluded; (2) screen-time variables are strongly target-associated and could be components of the unknown label definition, so they merit later ablation/leakage checks; (3) missingness differs mildly between train/test. There is no direct target proxy, duplicate-ID issue, or high-cardinality categorical.

## Reproducible baselines

In [9]:
prevalence = y.mean()
trivial_metrics = evaluate_predictions(y, np.full(len(y), prevalence))
fold_metrics, ml_oof = cross_validate_model(build_baseline_model(), X, y, CVConfig())
ml_metrics = evaluate_predictions(y, ml_oof)
pd.DataFrame([trivial_metrics, ml_metrics], index=['constant_prevalence', 'logistic_regression'])

,roc_auc,log_loss,accuracy_0.5
constant_prevalence,0.500000,0.602666,0.709424
logistic_regression,0.913785,0.338347,0.842285


In [10]:
fold_metrics

,fold,roc_auc,log_loss,accuracy_0.5
0,1,0.912733,0.340139,0.841481
1,2,0.913081,0.339826,0.841641
2,3,0.914453,0.337272,0.842212
3,4,0.914733,0.336172,0.843282
4,5,0.913933,0.338327,0.842811


The untuned logistic pipeline clearly beats the trivial baseline and is stable across folds. This justifies the locally validated baseline submission; no hyperparameter tuning or broad feature engineering is performed in Phase 1.